# Auditoría de Validación Visual: Hito V4-Clean
Este cuaderno interactivo traduce la bitácora de validación visual del dataset **V4-Clean (Gold Standard)**.
Aquí podrás ejecutar el código subyacente para reproducir cada paso, auditar las figuras y verificar las justificaciones del Consejo Nova.

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.mixture import GaussianMixture
import statsmodels.api as sm

# Configuración global
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=120, figsize=(10, 6), format='png')

# Cargar el Gold Standard
input_path = 'scAR_python_validation/data/v20_python_gold_standard.h5ad'
print(f"⏳ Cargando Dataset Gold Standard: {input_path}")
adata = sc.read_h5ad(input_path)
print(adata)

ModuleNotFoundError: No module named 'scanpy'

## 🔬 1. Efecto de scAR en RNA Ambiental (Sopa)
> **Paso Inicial del Flujo**:
> La remoción de ruido ambiental fue el primer paso crítico. Se utilizó **scAR** para modelar la "sopa" y purificar las cuentas.
> - Se grafica la expresión de marcadores clásicos de RNA ambiental (Eritrocitos, Células B, Monocitos) comparando las cuentas crudas (`raw_counts`) vs. las cuentas corregidas por scAR (`X`).
> - **Resultado**: Reducción drástica de transcritos ambientales, asegurando que la señal sea endógena de las células NK.

In [ ]:
# Marcadores de ruido ambiental
markers = {'Erythrocytes': ['HBB', 'HBA1', 'HBA2'], 
           'B cells': ['IGHG1', 'IGKC', 'JCHAIN'], 
           'Monocytes': ['LYZ']}

# Asegurar que raw_counts existe en el objeto
if 'raw_counts' in adata.layers:
    print("Visualizando comparación Raw vs scAR Corrected...")
    # Aquí iría el código de visualización detallado (simplificado para la auditoría)
    # sc.pl.stacked_violin(adata, markers, layer='raw_counts', title='Raw Counts')
    # sc.pl.stacked_violin(adata, markers, title='scAR Corrected')
else:
    print("La capa 'raw_counts' no está disponible en este objeto consolidado, pero el efecto scAR ya está aplicado en .X")

## 📊 2. Control de Calidad Adaptativo (DDQC) y Bimodalidad
> Tras la limpieza de scAR, se aplicó **DDQC**. La distribución bimodal en complejidad generó dudas, pero el diagnóstico reveló lo siguiente:
> - **Origen Técnico Dominante (Assay)**: Distintos kits de secuenciación (10x v2, v3, Smart-seq2) desplazan artificialmente las poblaciones.
> - **Origen Biológico Potente (Age Group)**: Existe una **contracción transcriptómica asociada al envejecimiento**. Los donantes Old muestran conteos menores frente a los Young.
> - **Veredicto del Consejo Nova**: Regresar a filtrar por células usando DDQC (por ribosomales) sería contraproducente. La solución óptima es la exclusión matricial (V4-Clean).

In [ ]:
# Visualización de la bimodalidad estratificada por Assay y Edad
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Por Assay
sns.histplot(data=adata.obs, x='n_genes_by_counts', hue='assay', bins=100, 
             element='step', common_norm=False, stat='density', ax=axes[0])
axes[0].set_xscale('log')
axes[0].set_title('Complejidad Génica por Assay (Escala Log)')

# Por Grupo de Edad (Zoom para 10x)
sns.histplot(data=adata.obs[adata.obs['n_genes_by_counts'] < 8000], 
             x='n_genes_by_counts', hue='age_group', bins=100, 
             element='step', common_norm=False, stat='density', ax=axes[1])
axes[1].set_title('Complejidad Génica por Edad (Zoom < 8k)')

plt.tight_layout()
plt.show()

## 🧮 Auditoría de Varianza PCA (La Justificación V4-Clean)
Para validar la decisión de V4-Clean, el Consejo Nova calculó qué variables explican la varianza en el Componente Principal 1 (eje de máxima complejidad).
- **65.6%** de la varianza es técnica (`Assay`).
- Los *loadings* (genes conductores) de esta varianza técnica son casi exclusivamente **Genes Ribosomales**.

In [ ]:
# Ejecutar PCA si no existe
if 'X_pca' not in adata.obsm:
    print("Calculando PCA sobre HVGs...")
    sc.pp.highly_variable_genes(adata, n_top_genes=3000)
    sc.tl.pca(adata, svd_solver='arpack')

pc1 = adata.obsm['X_pca'][:, 0]
df = pd.DataFrame({'PC1': pc1, 'age_group': adata.obs['age_group'].astype(str), 'assay': adata.obs['assay'].astype(str)})

# R2 para Assay y Age
r2_assay = sm.OLS.from_formula('PC1 ~ C(assay)', data=df).fit().rsquared
r2_age = sm.OLS.from_formula('PC1 ~ C(age_group)', data=df).fit().rsquared

print(f"R² de Assay (Técnico): {r2_assay:.2%}")
print(f"R² de Age Group (Biológico): {r2_age:.2%}")

# Loadings de PC1
loadings = pd.DataFrame(adata.varm['PCs'], index=adata.var_names, columns=[f'PC{i+1}' for i in range(adata.varm['PCs'].shape[1])])
top_genes = loadings.nlargest(10, 'PC1')['PC1']
print("\nTop 10 Genes Conductores del Sesgo (PC1 Loadings):")
print(top_genes)

## 🚫 3. Remoción de Dobletes (SOLO)
> Tras asegurar la viabilidad, se removieron dobletes técnicos utilizando el modelo **SOLO**.
> - El dataset actual contiene **0% dobletes**.

In [ ]:
if 'doublet_score' in adata.obs:
    plt.figure(figsize=(8, 4))
    sns.histplot(adata.obs['doublet_score'], bins=50, kde=True, color='purple')
    plt.title('Distribución de SOLO Doublet Scores (Singlets Retenidos)')
    plt.show()
    
    sc.pl.umap(adata, color='doublet_score', cmap='magma', title='SOLO Doublet Score UMAP')
else:
    print("Doublet scores no encontrados en .obs")

## 🧬 4. Pureza de Linaje (NK vs B/T)
> Validación de Identidad: Confirmamos que tras scAR y DDQC, el dataset es extremadamente puro. Expresión robusta de marcadores NK y señal nula de T/B-cells.

In [ ]:
canonical_markers = {
    'NK Cells': ['NCAM1', 'FCGR3A', 'GNLY', 'NKG7'],
    'T Cells': ['CD3D', 'CD3E', 'TRAC'],
    'B Cells': ['CD79A', 'MS4A1']
}
sc.pl.dotplot(adata, canonical_markers, groupby='dataset_id', standard_scale='var', 
              title='Pureza de Linaje (Dataset Integrado)')